# RHOAI Release Confidence Classifier

**Goal:** Given a RHOAI feature at planning freeze, predict the probability it ships in its committed phase (EA1, EA2, or GA).

**Training data:** RHOAI 3.4 + 3.5 committed features + FPDoR cycle snapshots — 319 rows total (280 shipped / 39 slipped).

**Algorithm:** Random Forest + SMOTE (inside each CV fold) + probability calibration.

**Output:** Per-feature shipping confidence (0–100%), baked into the release planner demo.

## 1. Setup

In [ ]:
import json, pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss, RocCurveDisplay
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
print('Libraries loaded.')

## 2. Training Data

Three sources, all in the same JSONL format:

| Source | Rows | Slipped | Notes |
|---|---|---|---|
| `3.4.jsonl` | 114 | 8 | RHOAI 3.4 committed features with full slip history |
| `3.5.json` | 205 | 4 | RHOAI 3.5 committed features |
| `fpdor_cycles_extended.jsonl` | 185 | 27 | Extracted from per-phase FPDoR snapshots (EA1/EA2/GA × 3.4+3.5) |

The FPDoR cycle snapshots have a `closure` field (`closed_by_t1`, `done_lag_after_t1`, `still_open_post_t1`) that tells us definitively whether a feature shipped at that phase freeze — without needing a matching feature record. The 27 slipped examples with `slip_count=0` are the key value: they teach the model to use FPDoR + RICE signals when there's no prior slip history.

In [ ]:
DATA_PATHS = [
    'path/to/3.4.jsonl',                   # update to your local path
    'path/to/3.5.json',                    # update to your local path
    'fpdor_cycles_extended.jsonl',         # in this repo
]

JIRA_PRI  = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1}
PHASE_ORD = {'EA1': 1, 'EA2': 2, 'GA': 3}

def load_rows():
    rows = []
    for path in DATA_PATHS:
        n = 0
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    r = json.loads(line)
                    r['_source'] = path.split('/')[-1]
                    rows.append(r)
                    n += 1
        print(f'  Loaded {n} rows from {path.split("/")[-1]}')
    return rows

rows = load_rows()
print(f'\nTotal rows: {len(rows)}')

## 3. Feature Extraction

12 features extracted from each training row at planning freeze:

| Feature | Source | Importance (v6) |
|---|---|---|
| `mandatory_pass_rate` | FPDoR checklist — mandatory items only | 19% |
| `slip_count` | # of prior release slips | 21% |
| `fpdor_pass_rate` | FPDoR overall pass % | 9% |
| `rt_pass` | FPDoR: Release Type item passed | 8% |
| `jira_priority` | Critical/High/Medium/Low → 4/3/2/1 | 8% |
| `rice` | RICE priority score | 7% |
| `criteria_pass_rate` | FPDoR criteria items pass % | 6% |
| `fpdor_passed_count` | # FPDoR items passed (absolute) | 6% |
| `committed_phase_ord` | EA1=1, EA2=2, GA=3 | 5% |
| `docs_pass` | FPDoR: Docs Impact item passed | 5% |
| `has_docs_component` | Feature has Documentation component | 3% |
| `component_encoded` | Primary component (label-encoded) | 3% |

In [ ]:
def extract_fpdor(fpdor):
    if not fpdor:
        return dict(pass_rate=0, mandatory_pass_rate=0, criteria_pass_rate=0,
                    passed_count=0, rt_pass=0, docs_pass=0)
    items      = fpdor.get('items', [])
    applicable = [i for i in items if i.get('state') != 'not-checked']
    mandatory  = [i for i in applicable if i.get('group') == 'mandatory']
    criteria   = [i for i in applicable if i.get('group') == 'criteria']
    def pr(lst): return sum(1 for i in lst if i.get('pass')) / len(lst) if lst else 0.0
    rt   = next((i for i in items if i['name'] == 'Release Type'), None)
    docs = next((i for i in items if i['name'] == 'Docs impact'),  None)
    ac   = fpdor.get('applicableCount', 1) or 1
    return dict(
        pass_rate           = fpdor.get('passedCount', 0) / ac,
        mandatory_pass_rate = pr(mandatory),
        criteria_pass_rate  = pr(criteria),
        passed_count        = fpdor.get('passedCount', 0),
        rt_pass             = int(rt['pass'] == True)   if rt   else 0,
        docs_pass           = int(docs['pass'] == True) if docs else 0,
    )

FEATURE_NAMES = [
    'fpdor_pass_rate', 'mandatory_pass_rate', 'criteria_pass_rate',
    'fpdor_passed_count', 'rt_pass', 'docs_pass',
    'rice', 'jira_priority', 'committed_phase_ord',
    'slip_count', 'has_docs_component', 'component_encoded',
]

def build_dataset(rows):
    committed = [r for r in rows
                 if r.get('committedPhase') and r['committedPhase'] not in (None, 'None', 'never')]
    le = LabelEncoder()
    le.fit([r.get('primaryComponent') or 'unknown' for r in committed])

    X_raw, y, keys = [], [], []
    for r in committed:
        label = 1 if r.get('deliveredPhase') == r.get('committedPhase') else 0
        sig   = extract_fpdor(r.get('fpdorAtFreeze'))
        rice  = r.get('priority', {}).get('rice')
        comp  = r.get('primaryComponent') or 'unknown'
        try:    comp_enc = le.transform([comp])[0]
        except  ValueError: comp_enc = 0
        X_raw.append([
            sig['pass_rate'], sig['mandatory_pass_rate'], sig['criteria_pass_rate'],
            sig['passed_count'], sig['rt_pass'], sig['docs_pass'],
            rice if rice is not None else np.nan,
            JIRA_PRI.get(r.get('priority', {}).get('jiraPriority', ''), 0),
            PHASE_ORD.get(r.get('committedPhase', 'GA'), 3),
            len(r.get('slips') or []),
            int(r.get('hasDocsComponent', False)),
            float(comp_enc),
        ])
        y.append(label)
        keys.append(r['key'])

    return np.array(X_raw, dtype=float), np.array(y), keys, le

X_raw, y, keys, comp_le = build_dataset(rows)
n_pos, n_neg = y.sum(), (y == 0).sum()
print(f'Dataset: {len(y)} samples  |  {n_pos} shipped ({n_pos/len(y)*100:.0f}%)  |  {n_neg} slipped ({n_neg/len(y)*100:.0f}%)')

## 4. Label Distribution

The dataset is heavily imbalanced — ~88% shipped, ~12% slipped. This is realistic (most committed features do ship), but means a naive model can reach 88% accuracy by predicting "ship" always. We correct for this with SMOTE inside each CV fold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Class distribution
axes[0].bar(['Shipped', 'Slipped'], [n_pos, n_neg], color=['#2e7d32', '#c62828'], alpha=0.85, width=0.5)
for i, v in enumerate([n_pos, n_neg]):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')
axes[0].set_title('Class Distribution (n=319)', fontweight='bold')
axes[0].set_ylabel('Features')
axes[0].set_ylim(0, 320)

# Source breakdown
sources = Counter(r['_source'] for r in rows)
labels = [s.replace('fpdor_cycles_extended.jsonl', 'FPDoR cycles\n(3.4+3.5)').replace('3.4.jsonl','3.4').replace('3.5.json','3.5') 
          for s in sources.keys()]
axes[1].bar(labels, sources.values(), color=['#1565c0', '#1976d2', '#42a5f5'], alpha=0.85, width=0.5)
for i, v in enumerate(sources.values()):
    axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold')
axes[1].set_title('Rows by Source', fontweight='bold')
axes[1].set_ylabel('Rows')

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Imputation

`rice` is missing for some features (~40 NaNs). We use MICE (Multiple Imputation by Chained Equations) — fits a Bayesian ridge regression per feature using all other features as predictors. Better than median imputation because it uses the joint distribution.

In [ ]:
mice_imp = IterativeImputer(max_iter=10, random_state=SEED, initial_strategy='median')
X_clean  = mice_imp.fit_transform(X_raw)

nan_before = np.isnan(X_raw).sum()
nan_after  = np.isnan(X_clean).sum()
print(f'NaN values: {nan_before} → {nan_after} (MICE imputation)')

## 6. Hyperparameter Tuning

GridSearchCV with 10-fold stratified CV searches across:
- `n_estimators`: 100, 200, 300
- `max_depth`: 3, 5, 7, 10
- `min_samples_leaf`: 3, 5, 10

SMOTE is inside the pipeline so it only sees training folds — no leakage.

In [ ]:
def make_rf(**kwargs):
    return RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1, **kwargs)

pipe = ImbPipeline([
    ('smote', SMOTE(random_state=SEED, k_neighbors=5)),
    ('rf',    make_rf()),
])
param_grid = {
    'rf__n_estimators':     [100, 200, 300],
    'rf__max_depth':        [3, 5, 7, 10],
    'rf__min_samples_leaf': [3, 5, 10],
}
grid = GridSearchCV(
    estimator  = pipe,
    param_grid = param_grid,
    cv         = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED),
    scoring    = 'roc_auc',
    n_jobs     = -1,
)
grid.fit(X_clean, y)
best_params = {k.replace('rf__', ''): v for k, v in grid.best_params_.items()}
print(f'Best params: {best_params}')
print(f'Best CV AUC: {grid.best_score_*100:.1f}%')

## 7. Cross-Validation Evaluation

10-fold stratified CV with the best hyperparameters. Each fold:
1. Split 319 samples → ~287 train / ~32 test
2. Apply SMOTE to training fold only → balance shipped/slipped
3. Train RF on balanced training fold
4. Evaluate on unaugmented test fold

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
smote = SMOTE(random_state=SEED, k_neighbors=5)
aucs, briers, f1s = [], [], []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X_clean, y), 1):
    X_tr, X_te = X_clean[tr_idx], X_clean[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    if len(np.unique(y_te)) < 2:
        print(f'  Fold {fold}: skipped (single class in test)')
        continue
    X_tr_b, y_tr_b = smote.fit_resample(X_tr, y_tr)
    rf = make_rf(**best_params)
    rf.fit(X_tr_b, y_tr_b)
    probs = rf.predict_proba(X_te)[:, 1]
    preds = rf.predict(X_te)
    aucs.append(roc_auc_score(y_te, probs))
    briers.append(brier_score_loss(y_te, probs))
    f1s.append(f1_score(y_te, preds, zero_division=0))
    print(f'  Fold {fold}: AUC={aucs[-1]*100:.1f}%  Brier={briers[-1]:.3f}  F1={f1s[-1]*100:.1f}%')

print(f'\nMean AUC:   {np.mean(aucs)*100:.1f}% ± {np.std(aucs)*100:.1f}%')
print(f'Mean Brier: {np.mean(briers):.3f} ± {np.std(briers):.3f}')
print(f'Mean F1:    {np.mean(f1s)*100:.1f}% ± {np.std(f1s)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# AUC per fold
axes[0].bar(range(1, len(aucs)+1), [a*100 for a in aucs], color='#1565c0', alpha=0.8)
axes[0].axhline(np.mean(aucs)*100, color='#c62828', linewidth=2, linestyle='--', label=f'Mean {np.mean(aucs)*100:.1f}%')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('AUC (%)')
axes[0].set_title('AUC per CV Fold', fontweight='bold')
axes[0].set_ylim(50, 105)
axes[0].legend()

# ROC curve (cross-val predictions)
probs_cv = cross_val_predict(
    make_rf(**best_params), X_clean, y,
    cv=StratifiedKFold(10, shuffle=True, random_state=SEED),
    method='predict_proba'
)[:, 1]
RocCurveDisplay.from_predictions(y, probs_cv, ax=axes[1], color='#1565c0')
axes[1].set_title('ROC Curve (10-fold CV)', fontweight='bold')
axes[1].plot([0,1],[0,1],'k--',alpha=0.4)

plt.tight_layout()
plt.savefig('cv_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Model Training + Calibration

Train on full dataset with SMOTE, then apply isotonic regression calibration. Calibration ensures "80% confidence" actually means 8 of 10 similar features ship — not just a relative ranking.

In [ ]:
# Train final RF on SMOTE-balanced full dataset
X_b, y_b = smote.fit_resample(X_clean, y)
print(f'After SMOTE: {Counter(y_b)}')

rf_final = make_rf(**best_params)
rf_final.fit(X_b, y_b)

# Calibrate probabilities
rf_calibrated = CalibratedClassifierCV(
    estimator = make_rf(**best_params),
    cv        = 5,
    method    = 'isotonic',
)
rf_calibrated.fit(X_b, y_b)
print('Model trained and calibrated.')

## 9. Calibration Check

A reliability diagram shows whether predicted probabilities match empirical frequencies. ECE (Expected Calibration Error) < 0.05 is the target for production use.

In [ ]:
probs_cal = cross_val_predict(
    CalibratedClassifierCV(make_rf(**best_params), cv=5, method='isotonic'),
    X_clean, y,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    method='predict_proba',
)[:, 1]

frac_pos, mean_pred = calibration_curve(y, probs_cal, n_bins=5)
ece = np.sum(np.abs(frac_pos - mean_pred) * (len(y) / 5)) / len(y)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(mean_pred, frac_pos, 'o-', color='#1565c0', label='Calibrated RF', linewidth=2, markersize=8)
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives (actual ship rate)')
ax.set_title(f'Reliability Diagram  (ECE = {ece:.3f})', fontweight='bold')
ax.legend()
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'ECE = {ece:.3f}  (target < 0.05; we are at {ece:.3f})')

## 10. Feature Importance

In [ ]:
importances = rf_final.feature_importances_
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1565c0' if importances[i] >= 0.10 else '#42a5f5' if importances[i] >= 0.05 else '#90caf9' 
          for i in order]
ax.barh([FEATURE_NAMES[i] for i in order[::-1]], 
        [importances[i]*100 for i in order[::-1]], 
        color=colors[::-1], alpha=0.85)
ax.set_xlabel('Importance (%)')
ax.set_title('Feature Importance — RF v6 (depth=10, n=319)', fontweight='bold')
for i, (feat_i, imp) in enumerate(zip(order[::-1], [importances[i] for i in order[::-1]])):
    ax.text(imp*100 + 0.2, i, f'{imp*100:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Confidence Score Distribution

Two populations of 3.6 scored features:
- **🤖 70 features** — have real slip history from 3.4/3.5 JSONL records. Cap at 95%.
- **~🤖 568 features** — CSV-only, `slip_count` assumed 0. Cap at 85%, minus FPDoR-completeness penalty (0–15pts).

The cap prevents overconfidence: without slip history, the model sees `slip_count=0` for every feature, which inflates scores by ~15pts on average.

In [ ]:
# Load pre-computed scores (from merge_scores.py output baked into index.html)
try:
    with open('merged_ml_scores.json') as f:
        scores_raw = json.load(f)
    # merged_ml_scores structure: {key: {score: float, source: str, phase: str}}
    scores = {k: v['score'] if isinstance(v, dict) else v for k, v in scores_raw.items()}
    print(f'Loaded {len(scores)} pre-computed scores.')

    # Load real keys (70 features with actual slip history)
    real_keys = set(k for k in scores if 'EA1' in scores_raw.get(k, {}).get('source', ''))
    # Approximate: use source == 'jsonl' or 'ea1' flag — adjust if needed

    real_scores  = [scores[k] for k in real_keys if k in scores]
    csv_scores   = [scores[k] for k in scores if k not in real_keys]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(real_scores, bins=20, range=(0,100), color='#2e7d32', alpha=0.8, edgecolor='white')
    axes[0].axvline(95, color='#c62828', linestyle='--', label='Cap: 95%')
    axes[0].axvline(np.mean(real_scores), color='#1565c0', linestyle='--', label=f'Mean: {np.mean(real_scores):.1f}%')
    axes[0].set_title(f'🤖 Real slip history (n={len(real_scores)}, cap 95%)', fontweight='bold')
    axes[0].set_xlabel('Confidence (%)'); axes[0].set_ylabel('Features'); axes[0].legend()

    axes[1].hist(csv_scores, bins=20, range=(0,100), color='#1565c0', alpha=0.8, edgecolor='white')
    axes[1].axvline(85, color='#c62828', linestyle='--', label='Cap: 85%')
    axes[1].axvline(np.mean(csv_scores), color='#2e7d32', linestyle='--', label=f'Mean: {np.mean(csv_scores):.1f}%')
    axes[1].set_title(f'~🤖 CSV-imputed (n={len(csv_scores)}, cap 85% - FPDoR penalty)', fontweight='bold')
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_ylabel('Features'); axes[1].legend()

    plt.tight_layout()
    plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
except FileNotFoundError:
    print('merged_ml_scores.json not found — run merge_scores.py first to generate inference output.')

## 12. Save Model

In [ ]:
out = {
    'rf':            rf_final,
    'rf_calibrated': rf_calibrated,
    'imputer':       mice_imp,
    'comp_le':       comp_le,
    'feature_names': FEATURE_NAMES,
    'best_params':   best_params,
    'cv_metrics':    {'auc': aucs, 'brier': briers, 'f1': f1s},
}
with open('models_v3.pkl', 'wb') as f:
    pickle.dump(out, f)
print('Saved → models_v3.pkl')
print('\nSummary:')
print(f'  Training set : {len(y)} rows  ({n_pos} shipped / {n_neg} slipped)')
print(f'  Best params  : {best_params}')
print(f'  CV AUC       : {np.mean(aucs)*100:.1f}% ± {np.std(aucs)*100:.1f}%')
print(f'  CV Brier     : {np.mean(briers):.3f}')
print(f'  ECE          : {ece:.3f}')

## 13. What's Next

| Path | Impact | Status |
|---|---|---|
| 3.6 ships → labeled rows | +~638 rows, push n to 950+ | Automatic when 3.6 GA |
| Jira API (via org-pulse) | Real `slip_count` for 568 ~🤖 features | Needs API access |
| SHAP per-feature explanations | Per-feature "why" in REASON column | Not yet built |

With ~950 labeled rows, AUC variance should drop from ±9.5% to ±4–5%, making the scores production-reliable across all feature types.

---
*Demo: https://github.com/yuvalluria/rhoai-release-planner — open index.html, no server needed.*